# Paddy production in Sri Lanka: comparing ensemble models
**Applied Machine Learning - hands-on activity**

The original statistic data came from the [Department of Census and Statistics, Sri Lanka](https://www.statistics.gov.lk/Agriculture/StaticalInformation/PaddyStatistics). I combined eight Excel sheets covering sown extent, harvested extent, yield and production for Maha and Yala, matching records by district, year and season. The combined CSV has 2,350 rows covering 1978–2025.

Here I compare Random Forest and XGBoost, then try LightGBM, CatBoost and a small RF+ANN experiment. I also made a basic prediction dashboard.

## 1. Load the data
The records cover districts, Maha/Yala seasons and years from 1978 to 2025. Production is in **thousand metric tons**. Upload the CSV when Colab asks for it.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

csv_file = Path('Sri_Lanka_Paddy_Production.csv')
if not csv_file.exists():
    try:
        from google.colab import files
        print('Please upload Sri_Lanka_Paddy_Production.csv')
        files.upload()
    except ImportError:
        raise FileNotFoundError('Put Sri_Lanka_Paddy_Production.csv beside this notebook.')

data = pd.read_csv(csv_file)
print('Original shape:', data.shape)
print(data.isna().sum())
data.head()

## 2. Prepare the inputs
I used district, season, year and sown extent. I left out harvested extent and average yield because those are known later and are closely tied to the production figure. Missing input/target rows are dropped.

In [ ]:
columns = ['District', 'Season', 'Year', 'Sown_Extent_Hectares']
target = 'Production_Thousand_Metric_Tons'
paddy = data[columns + [target]].dropna(subset=['Sown_Extent_Hectares', target]).copy()

X = paddy[columns]
y = paddy[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Keep encoding inside each pipeline so a new prediction uses the same preparation.
preparation = ColumnTransformer([('categories', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['District', 'Season'])], remainder='passthrough')

print(f'{len(paddy)} usable records | {len(X_train)} train | {len(X_test)} test')

## 3. Random Forest and XGBoost
Both models get the same train/test split. I used 300 trees and the settings I tried earlier. This is a small comparison, not a full hyperparameter search.

In [ ]:
rf = Pipeline([
    ('prepare', clone(preparation)),
    ('model', RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1))
])

xgb = Pipeline([
    ('prepare', clone(preparation)),
    ('model', XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05, objective='reg:squarederror', random_state=42, n_jobs=-1))
])

models = {'Random Forest': rf, 'XGBoost': xgb}
for name, model in models.items():
    model.fit(X_train, y_train)
    print(name, 'trained')

## 4. Compare performance
This is regression, so I used R², MAE and RMSE. Higher R² is better; lower errors are better.

In [ ]:
def scores(actual, predicted):
    return {
        'R2': r2_score(actual, predicted),
        'MAE': mean_absolute_error(actual, predicted),
        'RMSE': np.sqrt(mean_squared_error(actual, predicted))
    }

predictions = {name: model.predict(X_test) for name, model in models.items()}
results = pd.DataFrame({name: scores(y_test, pred) for name, pred in predictions.items()}).T
print(results.round(4))

## 5. Extra comparison: LightGBM and CatBoost
I wanted to see how LightGBM and CatBoost perform on the same data. I kept the same train/test split and encoding for a straightforward comparison.

In [ ]:
# Install these in Colab if they are not already available.
%pip -q install lightgbm catboost

from lightgbm import LGBMRegressor
# Keep the adapter in a module so saved models load in Streamlit.
# In Colab, upload model_wrappers.py alongside the CSV before running.
from model_wrappers import CompatibleCatBoost

extra_models = {
    'LightGBM': LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, verbosity=-1, random_state=42),
    'CatBoost': CompatibleCatBoost(iterations=300, learning_rate=0.05, depth=6, random_seed=42)
}

for name, regressor in extra_models.items():
    model = Pipeline([('prepare', clone(preparation)), ('model', regressor)])
    model.fit(X_train, y_train)
    models[name] = model
    predictions[name] = model.predict(X_test)
    results.loc[name] = scores(y_test, predictions[name])

print(results.sort_values('RMSE').round(4))

## 6. Optional hybrid: Random Forest + neural network
I tried a simple hybrid that averages RF and a small neural network's predictions (RF + ANN for tabular regression). The neural network needs scaled numeric inputs. This is an exploratory ensemble, not a new hybrid algorithm.

In [ ]:
nn_preparation = ColumnTransformer([
    ('categories', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['District', 'Season']),
    ('numbers', StandardScaler(), ['Year', 'Sown_Extent_Hectares'])
])

nn = Pipeline([
    ('prepare', nn_preparation),
    ('model', MLPRegressor(hidden_layer_sizes=(32, 16), max_iter=1200, early_stopping=True, random_state=42))
])

rf_ann = VotingRegressor([('forest', clone(rf)), ('neural_net', nn)])
rf_ann.fit(X_train, y_train)
models['RF + ANN'] = rf_ann
predictions['RF + ANN'] = rf_ann.predict(X_test)
results.loc['RF + ANN'] = scores(y_test, predictions['RF + ANN'])
print(results.sort_values('RMSE').round(4))

**Why not RF + ARIMA?** ARIMA is meant for an ordered time series. This dataset has separate district/season records, so an RF+ARIMA forecast would need a different time-based setup. I kept the tabular RF+ANN experiment instead.

## 7. Plot actual vs predicted
Points close to the dashed line are better predictions. I kept the axes the same across models to make them easier to compare.

In [ ]:
n_models = len(predictions)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
upper = max(y_test.max(), *(pred.max() for pred in predictions.values()))

for ax, (name, pred) in zip(axes.flat, predictions.items()):
    ax.scatter(y_test, pred, alpha=0.35, s=12)
    ax.plot([0, upper], [0, upper], 'r--', linewidth=1)
    ax.set(xlabel='Actual (thousand metric tons)', ylabel='Predicted', title=name, xlim=(0, upper), ylim=(0, upper))
    ax.grid(alpha=0.2)

for ax in axes.flat[n_models:]:
    ax.axis('off')

plt.tight_layout()
plt.show()

## 8. Streamlit dashboard

I built a separate Streamlit app for prediction and model comparison. It uses saved pipelines, so the input encoding stays the same as in training. The app and its instructions are included in the project ZIP.

In [ ]:
# Export the trained pipelines and results for use in the Streamlit app.
from pathlib import Path
import json
export_dir = Path('models')
export_dir.mkdir(exist_ok=True)
filenames = {
    'Random Forest': 'random_forest.joblib',
    'XGBoost': 'xgboost.joblib',
    'LightGBM': 'lightgbm.joblib',
    'CatBoost': 'catboost.joblib',
    'RF + ANN': 'rf_ann.joblib'
}
for name, filename in filenames.items():
    joblib.dump(models[name], export_dir / filename)

results.rename_axis('Model').reset_index().to_csv(export_dir / 'metrics.csv', index=False)
pd.DataFrame({'Actual': y_test.to_numpy(),
              **{name: pred for name, pred in predictions.items()}}
            ).to_csv(export_dir / 'test_predictions.csv', index=False)
(export_dir / 'metadata.json').write_text(json.dumps({
    'models': filenames,
    'districts': sorted(paddy.District.unique().tolist()),
    'seasons': sorted(paddy.Season.unique().tolist()),
    'year_min': int(paddy.Year.min()),
    'year_max': int(paddy.Year.max()),
    'rows': int(len(paddy)),
    'original_rows': int(len(data)),
    'split': 'Random 80/20, random_state=42',
    'target': target
}, indent=2))
print('Exported Streamlit model files to models/.')
print('For Windows, use the project ZIP and run train_models.py locally.')

## 9. Saving the models

The export cell above saves all five models and the metrics used by Streamlit. The standalone `train_models.py` in the project can recreate these files on Windows.

In [ ]:
print('Saved models:', ', '.join(filenames))

## 10. What I found
The table shows how Random Forest compares with three boosting methods and the RF+ANN combination. The Streamlit dashboard lets me select any of the five models for predictions.

**Limitations:** This is a random train/test split, not a test on future years. The comparison is exploratory, and rainfall, irrigation and soil information are missing. I would try a later-year test before using the model for real forecasting.